# QML-SleepNet — Stage 10 Model Robustness FINAL v1.1 — Record-Level Quartile Fix

## Role in the guide

This notebook implements the next missing roadmap stage:

**Stage 10 — Model Robustness**

The supplied roadmap asks for:

- **10.1 Cross-validation:** k-fold CV, inter-subject validation, unseen subjects.
- **10.2 Noise robustness:** Gaussian noise / SNR stress, degradation analysis, artifact segments.
- **10.3 Generalization:** different sleep stages, variable segment lengths, domain shift.
- **10.4 Ablation:** individual-feature removal, component comparison, sensitivity analysis.

## Frozen model under evaluation

Primary frozen Task-A system:

`Bridge + QT Angle-Rx + corrected Stage06 hybrid`

Each branch:

`raw probability → frozen temperature → frozen prior-corrected HMM`

Then:

`equal mean of the three branch logits → threshold 0.5`

Observed official-x result already frozen/scored:

**Accuracy = 90.1148%**

This Stage-10 notebook **cannot change that system**, regardless of robustness results.

## Hard rules

This notebook performs:

- **NO model training**
- **NO fine-tuning**
- **NO model selection**
- **NO threshold tuning**
- **NO HMM tuning**
- **NO ensemble-weight search**
- **NO new QML training**
- **NO new feature engineering**
- **NO attempt to improve headline metrics**

It is a downstream evaluation notebook only.

## Deliberately minimal Stage-10 implementation

### Cross-validation / unseen subjects
Reuses already-frozen guide CV and development evidence and the official unseen `x01–x35` result. No folds are retrained.

### Gaussian-noise stress
The Stage06 temporal branch consumes the exact frozen post-Stage02 ECG input. This notebook injects deterministic Gaussian noise **at that frozen model-input boundary** at:

- **20 dB SNR**
- **10 dB SNR**

The Bridge/QT HMM branches remain frozen while the perturbed Stage06 branch is re-inferred. Therefore this is correctly described as:

> **temporal ECG-channel robustness of the frozen final ensemble**

It is **not** claimed as full raw-signal end-to-end reprocessing robustness.

### Variable temporal coverage
The trained model expects 60-second inputs. We do not alter the architecture. Instead we evaluate:

- central **45 s** retained, remainder zero-masked;
- central **30 s** retained, remainder zero-masked.

This is reported as **temporal-coverage sensitivity**, not as a newly trained variable-length model.

### Artifact segments
A deterministic label-independent ECG-quality proxy is calculated from the already-filtered Stage02 signal. The high-artifact-proxy subset is evaluated separately.

### Generalization / domain shift
Report performance by:

- official Task-C A/B/C record category;
- apnea-burden quartile;
- record-length quartile.

### Sleep stages
The current frozen QML-SleepNet artifacts expose apnea/normal minute labels, not sleep-stage targets. Sleep-stage-specific performance is therefore reported as **NOT EVALUABLE FROM THE CURRENT PROJECT ARTIFACTS** rather than fabricated.

### Ablation
Two bounded analyses only:

1. frozen ensemble branch drop/single-branch comparisons;
2. small deterministic zero-out sensitivity for each QML8 and causal16 Stage06 input dimension.

No retraining is performed for ablation.

## Compute

Use a **GPU runtime**. The notebook is resume-safe per record and per robustness condition.

### v1.1 correction
Generalization quartiles are computed once per record and then mapped back to minute rows. This avoids weighting long records more heavily and avoids `qcut` label-count failures when repeated values collapse bin edges.

In [1]:
# Cell 1 — imports, Drive, paths, immutable artifact checks
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import gc, hashlib, json, math, os, random, time, warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

SEED = 20260914
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError(
        "Stage-10 perturbation inference is intended for a GPU runtime. "
        "Switch Colab to T4/L4/A100 and rerun."
    )

ROOT = Path("/content/drive/MyDrive/QML_SleepNet")
S2 = ROOT / "data/processed/stage02_preprocessed"

S4_FINAL = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX/04_quantum_module_v3_audited"
    / "final_fit/quantum_features_for_final_stage06.npz"
)
S5_FINAL = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX/05_causal_core_v1_2"
    / "final_fit/causal16_for_final_stage06.npz"
)

S6 = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX"
    / "05B_06_FINAL_GUIDE_CORRECTED_CONSOLIDATED_v1"
)
S6_MODEL = S6 / "final_fit/qml_sleepnet_final_guide_corrected.pt"
S6_THRESHOLD = S6 / "FINAL_OPERATING_THRESHOLD.json"
S6_FIVEFOLD = S6 / "five_fold_summary.json"

FINAL = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX"
    / "FINAL_FIXED_EQUAL_LOGIT_ENSEMBLE_v1"
)
FROZEN_X = FINAL / "FINAL_FIXED_EQUAL_LOGIT_X_PREDICTIONS_FROZEN_BEFORE_SCORING.npz"
X_METRICS = FINAL / "FINAL_FIXED_EQUAL_LOGIT_POSTHOC_X_METRICS.json"
X_PRED_CSV = FINAL / "FINAL_FIXED_EQUAL_LOGIT_POSTHOC_X_PREDICTIONS.csv"
DEV_DECISION = FINAL / "FIXED_EQUAL_LOGIT_DEVELOPMENT_DECISION.json"

TASKC = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX"
    / "STAGE06_TASKC_FINAL_ABC_FROM_FROZEN_TASKA_v1"
)
TASKC_RECORDS = TASKC / "TASKC_ABC_PER_RECORD_FINAL_RESULTS.csv"

OUT = (
    ROOT / "outputs/GUIDE_EXACT_METRICMAX"
    / "STAGE10_MODEL_ROBUSTNESS_FINAL_v1"
)
CACHE = OUT / "cache"
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

EXPECTED_FINAL_SHA = "f121a79191be00a28f33e06e7dec20cc689268b10a988c52e21284c90d1e2eef"

def sha256_file(path, chunk=1<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

for p in [
    S4_FINAL,S5_FINAL,S6_MODEL,S6_THRESHOLD,S6_FIVEFOLD,
    FROZEN_X,X_METRICS,X_PRED_CSV,DEV_DECISION,TASKC_RECORDS,
]:
    if not p.is_file():
        raise FileNotFoundError(p)

actual_sha=sha256_file(FROZEN_X)
if actual_sha != EXPECTED_FINAL_SHA:
    raise RuntimeError(f"Final frozen prediction drift: {actual_sha}")

print("DEVICE:", DEVICE)
print("Frozen Task-A SHA:", actual_sha)
print("Training performed in Stage10: NO")
print("Model selection performed in Stage10: NO")


Mounted at /content/drive
DEVICE: cuda
Frozen Task-A SHA: f121a79191be00a28f33e06e7dec20cc689268b10a988c52e21284c90d1e2eef
Training performed in Stage10: NO
Model selection performed in Stage10: NO


In [2]:
# Cell 2 — load frozen system; reproduce the locked 90.1148% official-x metric
EPS=1e-8
FS=100
EPOCH_SECONDS=60
EPOCH_SAMPLES=FS*EPOCH_SECONDS

fz=np.load(FROZEN_X,allow_pickle=False)
UID=np.asarray(fz["test_uids"]).astype(str)

BRIDGE_HMM=np.asarray(fz["bridge_hmm_posterior"],np.float64)
QT_HMM=np.asarray(fz["qt_hmm_posterior"],np.float64)
S6_HMM=np.asarray(fz["stage06_hmm_posterior"],np.float64)
S6_RAW=np.asarray(fz["stage06_raw_probability"],np.float64)
P_FINAL=np.asarray(fz["ensemble_hmm_logit_mean"],np.float64)

T_S6=float(np.asarray(fz["temperature_stage06"]).item())
CLASS_PRIOR=np.asarray(fz["class_prior"],np.float64)
PI=np.asarray(fz["initial_state_prior"],np.float64)
A=np.asarray(fz["transition_matrix"],np.float64)
HMM_LAMBDA=float(np.asarray(fz["hmm_lambda"]).item())
HARD_THRESHOLD=float(np.asarray(fz["hard_threshold"]).item())

pred_df=pd.read_csv(X_PRED_CSV)
pred_df["uid"]=pred_df["uid"].astype(str)

if not np.array_equal(pred_df["uid"].to_numpy(),UID):
    raise RuntimeError("Post-hoc x-label table UID order differs from frozen prediction artifact")

Y=np.asarray(pred_df["y_true"],np.int8)

def metrics(y,p,threshold=0.5):
    y=np.asarray(y,np.int8)
    p=np.asarray(p,np.float64)
    pred=(p>=threshold).astype(np.int8)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {
        "n":int(len(y)),
        "accuracy":float(accuracy_score(y,pred)),
        "balanced_accuracy":float(balanced_accuracy_score(y,pred)),
        "precision":float(precision_score(y,pred,zero_division=0)),
        "sensitivity":float(recall_score(y,pred,zero_division=0)),
        "specificity":float(tn/max(tn+fp,1)),
        "f1":float(f1_score(y,pred,zero_division=0)),
        "mcc":float(matthews_corrcoef(y,pred)),
        "auroc":float(roc_auc_score(y,p)) if len(np.unique(y))==2 else np.nan,
        "auprc":float(average_precision_score(y,p)) if len(np.unique(y))==2 else np.nan,
        "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
    }

M_FINAL=metrics(Y,P_FINAL,HARD_THRESHOLD)
expected=json.loads(X_METRICS.read_text())

if abs(M_FINAL["accuracy"]-float(expected["accuracy"]))>1e-12:
    raise RuntimeError("Frozen official-x metric reproduction failed")

print(json.dumps(M_FINAL,indent=2))
print("\nLOCKED OFFICIAL-X ACCURACY:",100*M_FINAL["accuracy"])
print("Final model changed: NO")


{
  "n": 17248,
  "accuracy": 0.9011479591836735,
  "balanced_accuracy": 0.9001158045109892,
  "precision": 0.8514808362369338,
  "sensitivity": 0.8958301512142967,
  "specificity": 0.9044014578076816,
  "f1": 0.8730926684034239,
  "mcc": 0.7929076491751258,
  "auroc": 0.9620666432037354,
  "auprc": 0.9427717592257047,
  "tn": 9678,
  "fp": 1023,
  "fn": 682,
  "tp": 5865
}

LOCKED OFFICIAL-X ACCURACY: 90.11479591836735
Final model changed: NO


In [3]:
# Cell 3 — load exact frozen QML8 / causal16 and reconstruct Stage06 model
qf=np.load(S4_FINAL,allow_pickle=False)
cf=np.load(S5_FINAL,allow_pickle=True)

Q_UID=np.asarray(qf["test_uids"]).astype(str)
Q_TEST=np.asarray(qf["vqc_angle8_test"],np.float32)

C_UID=np.asarray(cf["test_uids"]).astype(str)
C_TEST=np.asarray(cf["causal16_test"],np.float32)

if not np.array_equal(Q_UID,UID):
    raise RuntimeError("Stage04 test UID drift")
if not np.array_equal(C_UID,UID):
    raise RuntimeError("Stage05 test UID drift")
if Q_TEST.shape!=(len(UID),8):
    raise RuntimeError(Q_TEST.shape)
if C_TEST.shape!=(len(UID),16):
    raise RuntimeError(C_TEST.shape)

POOL=10
DROPOUT=0.4

class TemporalEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.c3=nn.Conv1d(1,64,3,padding=1)
        self.c5=nn.Conv1d(1,64,5,padding=2)
        self.c7=nn.Conv1d(1,64,7,padding=3)
        self.bn=nn.BatchNorm1d(192)
        self.pool=nn.MaxPool1d(POOL,POOL)
        self.bilstm=nn.LSTM(
            192,128,num_layers=2,batch_first=True,
            bidirectional=True,dropout=DROPOUT
        )
        self.attn=nn.MultiheadAttention(
            256,1,dropout=DROPOUT,batch_first=True
        )

    def forward(self,x):
        h=torch.cat([self.c3(x),self.c5(x),self.c7(x)],dim=1)
        h=self.pool(F.relu(self.bn(h))).transpose(1,2)
        h,_=self.bilstm(h)
        h,_=self.attn(h,h,h,need_weights=False)
        return torch.cat([h.mean(dim=1),h.max(dim=1).values],dim=1)

class GuideHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        self.temporal=TemporalEncoder()
        self.causal_gate=nn.Linear(16,16)
        self.fusion256=nn.Linear(536,256)
        self.fc128=nn.Linear(256,128)
        self.fc64=nn.Linear(128,64)
        self.out=nn.Linear(64,2)
        self.drop=nn.Dropout(DROPOUT)

    def fuse(self,temporal512,qml8,causal16):
        gate=torch.sigmoid(self.causal_gate(causal16))
        causal_gated=causal16*gate
        z=torch.cat([qml8,temporal512,causal_gated],dim=1)
        z=F.relu(self.fusion256(z))
        z=self.drop(z)
        z=F.relu(self.fc128(z))
        z=self.drop(z)
        z=F.relu(self.fc64(z))
        return self.out(z)

    def forward(self,ecg,qml8,causal16):
        t=self.temporal(ecg)
        return self.fuse(t,qml8,causal16),t

ck=torch.load(S6_MODEL,map_location="cpu",weights_only=False)
if ck.get("encoding")!="angle_rx":
    raise RuntimeError("Stage06 checkpoint encoding drift")
if ck.get("gate")!="causal16_to_gate16_on_causal_branch":
    raise RuntimeError("Stage06 checkpoint gate drift")

MODEL=GuideHybrid().to(DEVICE)
MODEL.load_state_dict(ck["state_dict"],strict=True)
MODEL.eval()

REC=np.asarray([u.rsplit(":",1)[0] for u in UID])
EP=np.asarray([int(u.rsplit(":",1)[1]) for u in UID],np.int64)

def load_record_ecg(rec, return_preclip=False):
    p=S2/f"{rec}_preprocessed.npz"
    if not p.is_file():
        raise FileNotFoundError(p)
    d=np.load(p,allow_pickle=False)
    ecg=np.asarray(d["ecg_filtered"],np.float32)
    fs=int(np.asarray(d["fs"]).item())
    ne=int(np.asarray(d["n_epochs"]).item())
    if fs!=FS or len(ecg)!=ne*EPOCH_SAMPLES:
        raise RuntimeError(f"{rec}: Stage02 ECG geometry mismatch")
    mu=float(ecg.mean())
    sd=float(ecg.std())
    if not np.isfinite(sd) or sd<1e-8:
        raise RuntimeError(f"{rec}: invalid ECG std")
    z=((ecg-mu)/sd).astype(np.float32).reshape(ne,EPOCH_SAMPLES)
    clipped=np.clip(z,-4.0,4.0).astype(np.float32)
    return (clipped,z) if return_preclip else clipped

@torch.no_grad()
def infer_arrays(X,Q,C,batch=256):
    out=[]
    for st in range(0,len(X),batch):
        en=min(st+batch,len(X))
        xb=torch.from_numpy(np.asarray(X[st:en],np.float32)).to(DEVICE)
        qb=torch.from_numpy(np.asarray(Q[st:en],np.float32)).to(DEVICE)
        cb=torch.from_numpy(np.asarray(C[st:en],np.float32)).to(DEVICE)
        logits,_=MODEL(xb,qb,cb)
        out.append(torch.softmax(logits,dim=1)[:,1].cpu().numpy())
    return np.concatenate(out).astype(np.float64)

print("Frozen Stage06 model reconstructed.")


Frozen Stage06 model reconstructed.


In [4]:
# Cell 4 — exact HMM/ensemble helpers copied from frozen final ensemble
def logit(p):
    p=np.clip(np.asarray(p,np.float64),EPS,1-EPS)
    return np.log(p)-np.log1p(-p)

def sigmoid(z):
    z=np.asarray(z,np.float64)
    out=np.empty_like(z)
    pos=z>=0
    out[pos]=1.0/(1.0+np.exp(-z[pos]))
    ez=np.exp(z[~pos])
    out[~pos]=ez/(1.0+ez)
    return out

def equal_logit_mean_n(*scores):
    S=np.column_stack([np.asarray(s,np.float64) for s in scores])
    return sigmoid(np.mean(np.column_stack([logit(S[:,i]) for i in range(S.shape[1])]),axis=1))

def temperature_scale(prob,T):
    p=np.clip(np.asarray(prob,float),EPS,1-EPS)
    z=logit(p)/float(T)
    return np.clip(sigmoid(z),EPS,1-EPS)

def contiguous_segments(indices,uids):
    indices=np.asarray(indices,np.int64)
    rec=np.asarray([u.rsplit(":",1)[0] for u in uids])
    ep=np.asarray([int(u.rsplit(":",1)[1]) for u in uids],np.int64)
    out=[]
    for r in np.unique(rec[indices]):
        rr=indices[rec[indices]==r]
        order=rr[np.argsort(ep[rr])]
        ee=ep[order]
        cuts=[0]+(np.where(np.diff(ee)!=1)[0]+1).tolist()+[len(order)]
        out.extend(order[a:b] for a,b in zip(cuts[:-1],cuts[1:]) if b>a)
    return out

def logsumexp(v):
    v=np.asarray(v,np.float64)
    m=np.max(v)
    return float(m+np.log(np.exp(v-m).sum()))

def forward_backward(prob,prior,pi,A):
    p=np.clip(np.asarray(prob,float),EPS,1-EPS)
    post=np.column_stack([1-p,p])
    logE=np.log(post)-HMM_LAMBDA*np.log(np.clip(prior,EPS,1))[None,:]
    logE-=np.max(logE,axis=1,keepdims=True)

    n=len(p)
    lpi=np.log(np.clip(pi,EPS,1))
    lA=np.log(np.clip(A,EPS,1))

    alpha=np.full((n,2),-np.inf)
    beta=np.full((n,2),-np.inf)
    alpha[0]=lpi+logE[0]

    for t in range(1,n):
        for s in range(2):
            alpha[t,s]=logE[t,s]+logsumexp(alpha[t-1]+lA[:,s])

    ll=logsumexp(alpha[-1])
    beta[-1]=0.0

    for t in range(n-2,-1,-1):
        for s in range(2):
            beta[t,s]=logsumexp(lA[s]+logE[t+1]+beta[t+1])

    gamma=np.exp(alpha+beta-ll)
    gamma/=gamma.sum(axis=1,keepdims=True)
    return gamma[:,1]

def decode_external(raw_prob,uids,T,prior,pi,A):
    pcal=temperature_scale(raw_prob,T)
    idx=np.arange(len(uids),dtype=np.int64)
    out=np.full(len(uids),np.nan,np.float64)
    for seg in contiguous_segments(idx,uids):
        out[seg]=forward_backward(pcal[seg],prior,pi,A)
    if not np.isfinite(out).all():
        raise RuntimeError("HMM decode incomplete")
    return out

# Verify clean frozen ensemble reconstruction.
reconstructed=equal_logit_mean_n(BRIDGE_HMM,QT_HMM,S6_HMM)
if np.max(np.abs(reconstructed-P_FINAL))>1e-12:
    raise RuntimeError("Frozen ensemble logit-mean reconstruction failed")

print("Frozen HMM/ensemble math reproduced exactly.")


Frozen HMM/ensemble math reproduced exactly.


In [5]:
# Cell 5 — Stage10.1 + Stage10.4: CV/unseen evidence + zero-training branch ablation
guide_cv=json.loads(S6_FIVEFOLD.read_text())
threshold_cv=json.loads(S6_THRESHOLD.read_text())
dev=json.loads(DEV_DECISION.read_text())

crossval_summary={
    "guide_stage06_fivefold_oof_raw_accuracy":float(guide_cv["oof"]["accuracy"]),
    "guide_stage06_learning_oof_accuracy_at_frozen_0415":float(
        threshold_cv["global_oof_selected_metrics"]["accuracy"]
    ),
    "final_fixed_ensemble_development_oof_accuracy":float(
        dev["fixed_ensemble"]["accuracy"]
    ),
    "official_unseen_x_accuracy":float(M_FINAL["accuracy"]),
    "notes":[
        "No cross-validation is rerun in Stage10.",
        "Existing guide CV / development evidence is reused.",
        "Official x01-x35 are the held-out unseen record set for the frozen final system.",
    ],
}
(OUT/"STAGE10_CV_UNSEEN_SUMMARY.json").write_text(
    json.dumps(crossval_summary,indent=2)
)

systems={
    "FINAL_3BRANCH":P_FINAL,
    "BRIDGE_ONLY":BRIDGE_HMM,
    "QT_ONLY":QT_HMM,
    "STAGE06_ONLY":S6_HMM,
    "DROP_BRIDGE_QT_PLUS_STAGE06":equal_logit_mean_n(QT_HMM,S6_HMM),
    "DROP_QT_BRIDGE_PLUS_STAGE06":equal_logit_mean_n(BRIDGE_HMM,S6_HMM),
    "DROP_STAGE06_BRIDGE_PLUS_QT":equal_logit_mean_n(BRIDGE_HMM,QT_HMM),
}

ablation=[]
for name,p in systems.items():
    m=metrics(Y,p,HARD_THRESHOLD)
    ablation.append({
        "system":name,
        **m,
        "accuracy_delta_vs_final_pp":100*(m["accuracy"]-M_FINAL["accuracy"]),
    })

ablation=pd.DataFrame(ablation).sort_values("accuracy",ascending=False)
ablation.to_csv(OUT/"STAGE10_BRANCH_ABLATION.csv",index=False)

print("CROSS-VALIDATION / UNSEEN SUMMARY")
print(json.dumps(crossval_summary,indent=2))
print("\nFROZEN BRANCH ABLATIONS — evaluation only")
display(ablation)


CROSS-VALIDATION / UNSEEN SUMMARY
{
  "guide_stage06_fivefold_oof_raw_accuracy": 0.8324619632262233,
  "guide_stage06_learning_oof_accuracy_at_frozen_0415": 0.838160136286201,
  "final_fixed_ensemble_development_oof_accuracy": 0.8881513246783763,
  "official_unseen_x_accuracy": 0.9011479591836735,
  "notes": [
    "No cross-validation is rerun in Stage10.",
    "Existing guide CV / development evidence is reused.",
    "Official x01-x35 are the held-out unseen record set for the frozen final system."
  ]
}

FROZEN BRANCH ABLATIONS — evaluation only


,system,n,accuracy,balanced_accuracy,precision,sensitivity,specificity,f1,mcc,auroc,auprc,tn,fp,fn,tp,accuracy_delta_vs_final_pp
6,DROP_STAGE06_BRIDGE_PLUS_QT,17248,0.905032,0.901883,0.864764,0.888804,0.914961,0.876619,0.799650,0.959930,0.943571,9791,910,728,5819,0.388451
2,QT_ONLY,17248,0.903873,0.895256,0.884053,0.859478,0.931034,0.871592,0.795002,0.960364,0.941189,9963,738,920,5627,0.272495
0,FINAL_3BRANCH,17248,0.901148,0.900116,0.851481,0.895830,0.904401,0.873093,0.792908,0.962067,0.942772,9678,1023,682,5865,0.000000
4,DROP_BRIDGE_QT_PLUS_STAGE06,17248,0.899583,0.895623,0.859489,0.879181,0.912064,0.869224,0.787876,0.958968,0.935022,9760,941,791,5756,-0.156540
1,BRIDGE_ONLY,17248,0.893959,0.897761,0.825649,0.913548,0.881974,0.867377,0.782185,0.957895,0.939676,9438,1263,566,5981,-0.718924
5,DROP_QT_BRIDGE_PLUS_STAGE06,17248,0.891929,0.895029,0.824983,0.907897,0.882161,0.864456,0.777398,0.960041,0.939145,9440,1261,603,5944,-0.921846
3,STAGE06_ONLY,17248,0.880914,0.883038,0.812665,0.891859,0.874217,0.850422,0.754099,0.946001,0.913128,9355,1346,708,5839,-2.023423


In [6]:
# Cell 6 — Stage10.3 + artifact-segment audit using existing frozen final predictions
taskc=pd.read_csv(TASKC_RECORDS)
taskc["record_name"]=taskc["record_name"].astype(str)

# Per-minute frame with record category and record statistics.
minute=pd.DataFrame({
    "uid":UID,
    "record_name":REC,
    "epoch_idx":EP,
    "y":Y,
    "p":P_FINAL,
})
minute=minute.merge(
    taskc[[
        "record_name","scorable_minutes","official_annotated_minutes_full",
        "true_apnea_minutes_full","taskc_true_class_full"
    ]],
    on="record_name",
    how="left",
    validate="many_to_one",
)
if minute["taskc_true_class_full"].isna().any():
    raise RuntimeError("Task-C record metadata merge incomplete")

minute["apnea_burden"]=(
    minute["true_apnea_minutes_full"]/
    minute["official_annotated_minutes_full"].clip(lower=1)
)
# Build quartiles at the RECORD level, then map them back to minutes.
# Using qcut directly on minute rows would over-weight long records and can fail
# when repeated record-level values collapse quantile edges.
record_meta = (
    minute[["record_name","apnea_burden","scorable_minutes"]]
    .drop_duplicates("record_name")
    .sort_values("record_name")
    .reset_index(drop=True)
)

def robust_record_qbin(series, prefix):
    codes = pd.qcut(
        series.astype(float),
        q=4,
        labels=False,
        duplicates="drop",
    )
    n_bins = int(codes.max()) + 1 if codes.notna().any() else 0
    return codes.map(
        lambda x: f"{prefix}{int(x)+1}_of_{n_bins}" if pd.notna(x) else "NA"
    )

record_meta["burden_quartile"] = robust_record_qbin(
    record_meta["apnea_burden"], "Q"
)
record_meta["length_quartile"] = robust_record_qbin(
    record_meta["scorable_minutes"], "Q"
)

minute = minute.merge(
    record_meta[["record_name","burden_quartile","length_quartile"]],
    on="record_name",
    how="left",
    validate="many_to_one",
)

def grouped_metrics(frame,col):
    rows=[]
    for val,g in frame.groupby(col,observed=True):
        m=metrics(g["y"].to_numpy(),g["p"].to_numpy(),HARD_THRESHOLD)
        rows.append({"grouping":col,"group":str(val),**m})
    return rows

gen_rows=[]
for col in ["taskc_true_class_full","burden_quartile","length_quartile"]:
    gen_rows.extend(grouped_metrics(minute,col))

gen=pd.DataFrame(gen_rows)
gen.to_csv(OUT/"STAGE10_GENERALIZATION_DOMAIN_SHIFT.csv",index=False)

# Label-independent Stage02 artifact proxy.
quality_rows=[]
for rec in sorted(np.unique(REC)):
    idx=np.where(REC==rec)[0]
    eps=EP[idx]
    _,preclip=load_record_ecg(rec,return_preclip=True)

    X=preclip[eps]
    clip_frac=np.mean(np.abs(X)>4.0,axis=1)
    dX=np.diff(X,axis=1)
    diff_rms=np.sqrt(np.mean(np.square(dX),axis=1))
    flat_frac=np.mean(np.abs(dX)<1e-4,axis=1)

    for row,cf,dr,ff in zip(idx,clip_frac,diff_rms,flat_frac):
        quality_rows.append({
            "row":int(row),
            "uid":UID[row],
            "clip_fraction_preclip":float(cf),
            "diff_rms":float(dr),
            "flat_fraction":float(ff),
        })

quality=pd.DataFrame(quality_rows).sort_values("row").reset_index(drop=True)
if quality["row"].tolist()!=list(range(len(UID))):
    raise RuntimeError("Quality table row alignment failed")

diff_cut=float(quality["diff_rms"].quantile(0.75))
flat_cut=float(quality["flat_fraction"].quantile(0.75))

artifact_mask=(
    (quality["clip_fraction_preclip"].to_numpy()>0)
    | (quality["diff_rms"].to_numpy()>=diff_cut)
    | (quality["flat_fraction"].to_numpy()>=flat_cut)
)

artifact_metrics=metrics(
    Y[artifact_mask],
    P_FINAL[artifact_mask],
    HARD_THRESHOLD,
)
artifact_report={
    "definition":
        "heuristic Stage02 artifact-prone subset: any preclip |z|>4 OR top-quartile diff_rms OR top-quartile flat_fraction",
    "label_independent_definition":True,
    "n":int(artifact_mask.sum()),
    "fraction":float(artifact_mask.mean()),
    "diff_rms_q75":diff_cut,
    "flat_fraction_q75":flat_cut,
    "metrics":artifact_metrics,
    "important_limitation":
        "This is a deterministic ECG-quality stress proxy, not a clinically validated artifact annotation.",
}
(OUT/"STAGE10_ARTIFACT_SEGMENT_REPORT.json").write_text(
    json.dumps(artifact_report,indent=2)
)
quality.to_csv(OUT/"STAGE10_ECG_QUALITY_PROXY.csv",index=False)

unsupported={
    "different_sleep_stages":{
        "status":"NOT_EVALUABLE_FROM_CURRENT_PROJECT_ARTIFACTS",
        "reason":
            "The frozen prediction/target chain exposes Apnea-vs-Normal minute targets and record A/B/C categories; no sleep-stage target is available in the current final artifact contract.",
        "action":"Do not fabricate sleep-stage performance."
    }
}
(OUT/"STAGE10_UNSUPPORTED_GUIDE_ITEMS.json").write_text(
    json.dumps(unsupported,indent=2)
)

print("GENERALIZATION / DOMAIN-SHIFT TABLE")
display(gen)
print("\nARTIFACT-PROXY REPORT")
print(json.dumps(artifact_report,indent=2))
print("\nSleep-stage-specific evaluation: NOT FABRICATED / NOT EVALUABLE.")


GENERALIZATION / DOMAIN-SHIFT TABLE


,grouping,group,n,accuracy,balanced_accuracy,precision,sensitivity,specificity,f1,mcc,auroc,auprc,tn,fp,fn,tp
0,taskc_true_class_full,A,10182,0.878216,0.871052,0.901447,0.901447,0.840658,0.901447,0.742104,0.946540,0.965065,3271,620,620,5671
1,taskc_true_class_full,B,2471,0.867260,0.836565,0.410148,0.798354,0.874776,0.541899,0.509485,0.901879,0.604599,1949,279,49,194
2,taskc_true_class_full,C,4595,0.970185,0.486469,0.000000,0.000000,0.972938,0.000000,-0.008871,0.672699,0.004887,4458,124,13,0
3,burden_quartile,Q1_of_4,4121,0.967969,0.485044,0.000000,0.000000,0.970088,0.000000,-0.008206,0.690175,0.004112,3989,123,9,0
4,burden_quartile,Q2_of_4,4488,0.907308,0.896208,0.629792,0.880551,0.911864,0.734355,0.694095,0.951839,0.816254,3497,338,78,575
5,burden_quartile,Q3_of_4,4023,0.878697,0.874812,0.848729,0.938613,0.811011,0.891411,0.759722,0.949196,0.956055,1532,357,131,2003
6,burden_quartile,Q4_of_4,4616,0.855069,0.819653,0.941294,0.876300,0.763006,0.907635,0.581259,0.907721,0.974744,660,205,464,3287
7,length_quartile,Q1_of_4,4067,0.954758,0.909858,0.579832,0.858921,0.960795,0.692308,0.683948,0.961470,0.847098,3676,150,34,207
8,length_quartile,Q2_of_4,4385,0.916762,0.911271,0.953160,0.856925,0.965617,0.902485,0.833813,0.979785,0.974963,2331,83,282,1689
9,length_quartile,Q3_of_4,4063,0.870047,0.875209,0.810556,0.931744,0.818674,0.866935,0.748026,0.949600,0.939547,1815,402,126,1720



ARTIFACT-PROXY REPORT
{
  "definition": "heuristic Stage02 artifact-prone subset: any preclip |z|>4 OR top-quartile diff_rms OR top-quartile flat_fraction",
  "label_independent_definition": true,
  "n": 16812,
  "fraction": 0.974721706864564,
  "diff_rms_q75": 0.7627919465303421,
  "flat_fraction_q75": 0.002333722287047841,
  "metrics": {
    "n": 16812,
    "accuracy": 0.9025695931477516,
    "balanced_accuracy": 0.9013941154359807,
    "precision": 0.8581946694154028,
    "sensitivity": 0.8962017530370598,
    "specificity": 0.9065864778349015,
    "f1": 0.8767865202346924,
    "mcc": 0.7967897662385018,
    "auroc": 0.962922460539527,
    "auprc": 0.9454445010049047,
    "tn": 9346,
    "fp": 963,
    "fn": 675,
    "tp": 5828
  },
  "important_limitation": "This is a deterministic ECG-quality stress proxy, not a clinically validated artifact annotation."
}

Sleep-stage-specific evaluation: NOT FABRICATED / NOT EVALUABLE.


In [8]:
# Cell 7 — exact frozen Stage06 clean reproduction check before perturbation
# Verify the reloaded model + exact Stage02 input recreates the frozen raw Stage06 predictions.
# One full record is sufficient as a fail-closed implementation check.

rec0=sorted(np.unique(REC))[0]
idx0=np.where(REC==rec0)[0]
eps0=EP[idx0]
z0=load_record_ecg(rec0)
X0=z0[eps0,None,:].astype(np.float32)

p0=infer_arrays(X0,Q_TEST[idx0],C_TEST[idx0],batch=256)
max_abs=float(np.max(np.abs(p0-S6_RAW[idx0])))
mean_abs=float(np.mean(np.abs(p0-S6_RAW[idx0])))

repro={
    "record":rec0,
    "n":int(len(idx0)),
    "max_abs_probability_difference":max_abs,
    "mean_abs_probability_difference":mean_abs,
    "pass":bool(max_abs<=1e-4),
}
(OUT/"STAGE10_STAGE06_RELOAD_REPRO_CHECK.json").write_text(
    json.dumps(repro,indent=2)
)

print(json.dumps(repro,indent=2))
if not repro["pass"]:
    raise RuntimeError(
        "Reloaded Stage06 inference does not reproduce the frozen branch closely enough. "
        "Do not run robustness perturbations until this is resolved."
    )


{
  "record": "x01",
  "n": 522,
  "max_abs_probability_difference": 2.9802322387695312e-08,
  "mean_abs_probability_difference": 2.283702864957495e-10,
  "pass": true
}


In [10]:
# Cell 8 — perturbation inference: SNR 20/10 dB + 45s/30s temporal coverage
CONDITIONS={
    "SNR20_DB":{"kind":"snr","snr_db":20.0},
    "SNR10_DB":{"kind":"snr","snr_db":10.0},
    "CENTRAL_45S":{"kind":"coverage","seconds":45},
    "CENTRAL_30S":{"kind":"coverage","seconds":30},
}

def condition_seed(rec,name):
    rec_num=int("".join(ch for ch in rec if ch.isdigit()) or 0)
    name_code=sum(ord(c) for c in name)
    return int(SEED + 1009*rec_num + 17*name_code)

def perturb(X,spec,rec,name):
    X=np.asarray(X,np.float32)

    if spec["kind"]=="snr":
        snr=float(spec["snr_db"])
        rng=np.random.default_rng(condition_seed(rec,name))
        power=np.mean(np.square(X),axis=2,keepdims=True)
        noise_power=np.maximum(power,1e-8)/(10.0**(snr/10.0))
        noise=rng.normal(0.0,1.0,size=X.shape).astype(np.float32)
        noise*=np.sqrt(noise_power).astype(np.float32)
        return (X+noise).astype(np.float32)

    if spec["kind"]=="coverage":
        sec=int(spec["seconds"])
        keep=sec*FS
        if keep<=0 or keep>EPOCH_SAMPLES:
            raise ValueError(spec)
        start=(EPOCH_SAMPLES-keep)//2
        out=np.zeros_like(X)
        out[:,:,start:start+keep]=X[:,:,start:start+keep]
        return out

    raise ValueError(spec)

def infer_condition(name,spec):
    full=np.full(len(UID),np.nan,np.float64)

    for ri,rec in enumerate(sorted(np.unique(REC)),1):
        idx=np.where(REC==rec)[0]
        cp=CACHE/f"{name}__{rec}.npz"

        if cp.is_file():
            z=np.load(cp,allow_pickle=False)
            if np.array_equal(np.asarray(z["rows"],np.int64),idx):
                p=np.asarray(z["raw_probability"],np.float64)
                if len(p)==len(idx) and np.isfinite(p).all():
                    full[idx]=p
                    print(f"{name}: {rec} RESUME ({ri}/35)")
                    continue
            raise RuntimeError(f"Stale/incompatible cache: {cp}")

        eps=EP[idx]
        clean=load_record_ecg(rec)[eps,None,:].astype(np.float32)
        Xp=perturb(clean,spec,rec,name)

        p=infer_arrays(
            Xp,
            Q_TEST[idx],
            C_TEST[idx],
            batch=256,
        )

        np.savez_compressed(
            cp,
            rows=idx.astype(np.int64),
            raw_probability=p.astype(np.float32),
        )
        full[idx]=p

        print(f"{name}: {rec} DONE ({ri}/35)")

        del clean,Xp,p
        gc.collect()
        torch.cuda.empty_cache()

    if not np.isfinite(full).all():
        raise RuntimeError(f"{name}: incomplete inference")

    return full

robust_rows=[]

for name,spec in CONDITIONS.items():
    print("\n"+"="*110)
    print("ROBUSTNESS CONDITION:",name,spec)
    print("="*110)

    raw=infer_condition(name,spec)

    s6_hmm=decode_external(
        raw,UID,T_S6,CLASS_PRIOR,PI,A
    )

    # Temporal-channel perturbation:
    # bridge + QT remain their frozen clean HMM branches.
    ens=equal_logit_mean_n(
        BRIDGE_HMM,
        QT_HMM,
        s6_hmm,
    )

    ms6=metrics(Y,s6_hmm,HARD_THRESHOLD)
    mens=metrics(Y,ens,HARD_THRESHOLD)

    robust_rows.append({
        "condition":name,
        "stage06_accuracy":ms6["accuracy"],
        "stage06_balanced_accuracy":ms6["balanced_accuracy"],
        "stage06_f1":ms6["f1"],
        "stage06_auroc":ms6["auroc"],
        "final_ensemble_accuracy":mens["accuracy"],
        "final_ensemble_balanced_accuracy":mens["balanced_accuracy"],
        "final_ensemble_f1":mens["f1"],
        "final_ensemble_auroc":mens["auroc"],
        "final_ensemble_accuracy_delta_vs_clean_pp":
            100*(mens["accuracy"]-M_FINAL["accuracy"]),
    })

    print("Stage06 perturbed:",json.dumps(ms6,indent=2))
    print("Final ensemble with perturbed temporal branch:",json.dumps(mens,indent=2))

robust=pd.DataFrame(robust_rows)
robust.to_csv(OUT/"STAGE10_PERTURBATION_ROBUSTNESS.csv",index=False)
display(robust)



ROBUSTNESS CONDITION: SNR20_DB {'kind': 'snr', 'snr_db': 20.0}
SNR20_DB: x01 DONE (1/35)
SNR20_DB: x02 DONE (2/35)
SNR20_DB: x03 DONE (3/35)
SNR20_DB: x04 DONE (4/35)
SNR20_DB: x05 DONE (5/35)
SNR20_DB: x06 DONE (6/35)
SNR20_DB: x07 DONE (7/35)
SNR20_DB: x08 DONE (8/35)
SNR20_DB: x09 DONE (9/35)
SNR20_DB: x10 DONE (10/35)
SNR20_DB: x11 DONE (11/35)
SNR20_DB: x12 DONE (12/35)
SNR20_DB: x13 DONE (13/35)
SNR20_DB: x14 DONE (14/35)
SNR20_DB: x15 DONE (15/35)
SNR20_DB: x16 DONE (16/35)
SNR20_DB: x17 DONE (17/35)
SNR20_DB: x18 DONE (18/35)
SNR20_DB: x19 DONE (19/35)
SNR20_DB: x20 DONE (20/35)
SNR20_DB: x21 DONE (21/35)
SNR20_DB: x22 DONE (22/35)
SNR20_DB: x23 DONE (23/35)
SNR20_DB: x24 DONE (24/35)
SNR20_DB: x25 DONE (25/35)
SNR20_DB: x26 DONE (26/35)
SNR20_DB: x27 DONE (27/35)
SNR20_DB: x28 DONE (28/35)
SNR20_DB: x29 DONE (29/35)
SNR20_DB: x30 DONE (30/35)
SNR20_DB: x31 DONE (31/35)
SNR20_DB: x32 DONE (32/35)
SNR20_DB: x33 DONE (33/35)
SNR20_DB: x34 DONE (34/35)
SNR20_DB: x35 DONE (35/35)


,condition,stage06_accuracy,stage06_balanced_accuracy,stage06_f1,stage06_auroc,final_ensemble_accuracy,final_ensemble_balanced_accuracy,final_ensemble_f1,final_ensemble_auroc,final_ensemble_accuracy_delta_vs_clean_pp
0,SNR20_DB,0.851693,0.839092,0.801089,0.920424,0.901322,0.897973,0.871818,0.957818,0.017393
1,SNR10_DB,0.846243,0.852458,0.812606,0.922885,0.897727,0.897122,0.869120,0.957092,-0.342069
2,CENTRAL_45S,0.878189,0.882087,0.848446,0.945911,0.900916,0.900463,0.873173,0.962016,-0.023191
3,CENTRAL_30S,0.875174,0.880784,0.846115,0.945132,0.900626,0.900407,0.872962,0.962059,-0.052180


In [14]:
# Cell 9 — bounded individual-feature zero-out sensitivity (NO retraining)
# A small deterministic subset is enough for sensitivity ranking and avoids turning
# Stage10 into another expensive experiment.

N_SENS=min(256,len(UID))
SENS_ROWS=np.unique(
    np.linspace(0,len(UID)-1,N_SENS,dtype=np.int64)
)

def build_ecg_rows(rows):
    rows=np.asarray(rows,np.int64)
    X=np.empty((len(rows),1,EPOCH_SAMPLES),np.float32)

    pos={int(r):i for i,r in enumerate(rows)}
    recs=np.unique(REC[rows])

    for rec in recs:
        rr=rows[REC[rows]==rec]
        z=load_record_ecg(rec)
        for r in rr:
            X[pos[int(r)],0]=z[EP[r]]
    return X

Xsens=build_ecg_rows(SENS_ROWS)
Qsens=Q_TEST[SENS_ROWS].copy()
Csens=C_TEST[SENS_ROWS].copy()

clean_sens=infer_arrays(Xsens,Qsens,Csens,batch=128)
frozen_sens=S6_RAW[SENS_ROWS]

if np.max(np.abs(clean_sens-frozen_sens))>1e-4:
    raise RuntimeError("Sensitivity subset clean reproduction failed")

sens=[]

for j in range(8):
    Qx=Qsens.copy()
    Qx[:,j]=0.0

    p=infer_arrays(
        Xsens,
        Qx,
        Csens,
        batch=128
    )

    sens.append({
        "branch":"QML8",
        "feature_index":j,
        "mean_abs_probability_change":float(
            np.mean(np.abs(p-clean_sens))
        ),
        "max_abs_probability_change":float(
            np.max(np.abs(p-clean_sens))
        ),
    })

for j in range(16):
    Cx=Csens.copy()
    Cx[:,j]=0.0

    p=infer_arrays(
        Xsens,
        Qsens,
        Cx,
        batch=128
    )

    sens.append({
        "branch":"CAUSAL16",
        "feature_index":j,
        "mean_abs_probability_change":float(
            np.mean(np.abs(p-clean_sens))
        ),
        "max_abs_probability_change":float(
            np.max(np.abs(p-clean_sens))
        ),
    })

sens=pd.DataFrame(sens).sort_values(
    "mean_abs_probability_change",
    ascending=False,
).reset_index(drop=True)

sens.to_csv(
    OUT/"STAGE10_INDIVIDUAL_FEATURE_ZEROOUT_SENSITIVITY.csv",
    index=False,
)

display(sens)

del Xsens,Qsens,Csens,clean_sens
gc.collect()
torch.cuda.empty_cache()

,branch,feature_index,mean_abs_probability_change,max_abs_probability_change
0,CAUSAL16,0,0.220672,0.772057
1,CAUSAL16,1,0.143301,0.699562
2,CAUSAL16,2,0.089670,0.537980
3,CAUSAL16,4,0.065887,0.528763
4,QML8,7,0.061941,0.244960
5,QML8,6,0.046251,0.192777
6,CAUSAL16,6,0.038997,0.184800
7,CAUSAL16,7,0.036677,0.207540
8,QML8,1,0.036578,0.169443
9,QML8,5,0.033671,0.185852


In [15]:
# Cell 10 — final Stage10 manifest / close robustness stage
final_manifest={
    "schema":"QML_SleepNet_STAGE10_MODEL_ROBUSTNESS_FINAL_v1",
    "stage":"Stage 10 — Model Robustness",
    "primary_frozen_model":
        "Bridge + QT Angle-Rx + corrected Stage06; frozen temperature/HMM per branch; equal logit mean",
    "primary_frozen_prediction_sha256":EXPECTED_FINAL_SHA,
    "training_performed":False,
    "model_selection_performed":False,
    "threshold_tuning_performed":False,
    "ensemble_weight_tuning_performed":False,
    "headline_model_modified":False,
    "official_x_labels_role":
        "evaluation only; official x had already been opened before Stage10",
    "stage10_1_cross_validation":{
        "status":"COMPLETE_BY_REUSE",
        "artifact":"STAGE10_CV_UNSEEN_SUMMARY.json",
    },
    "stage10_2_noise_robustness":{
        "status":"COMPLETE",
        "conditions":["SNR20_DB","SNR10_DB"],
        "scope":
            "Gaussian noise added at exact frozen post-Stage02 ECG model-input boundary; temporal Stage06 branch perturbed, Bridge/QT branches held frozen",
        "artifact":"STAGE10_PERTURBATION_ROBUSTNESS.csv",
    },
    "stage10_2_artifact_segments":{
        "status":"COMPLETE_WITH_HEURISTIC_PROXY",
        "artifact":"STAGE10_ARTIFACT_SEGMENT_REPORT.json",
        "limitation":
            "quality proxy is deterministic but not a clinically annotated artifact label",
    },
    "stage10_3_generalization":{
        "domain_shift":"COMPLETE: Task-C class, apnea-burden quartile, record-length quartile",
        "variable_temporal_coverage":"COMPLETE: central 45s and 30s masking stress",
        "different_sleep_stages":
            "NOT EVALUABLE: no sleep-stage target in current frozen project artifact contract",
    },
    "stage10_4_ablation":{
        "branch_ablation":"COMPLETE",
        "individual_feature_sensitivity":
            "COMPLETE: bounded QML8/causal16 zero-out probability sensitivity, no retraining",
    },
    "outputs":[
        "STAGE10_CV_UNSEEN_SUMMARY.json",
        "STAGE10_BRANCH_ABLATION.csv",
        "STAGE10_GENERALIZATION_DOMAIN_SHIFT.csv",
        "STAGE10_ECG_QUALITY_PROXY.csv",
        "STAGE10_ARTIFACT_SEGMENT_REPORT.json",
        "STAGE10_UNSUPPORTED_GUIDE_ITEMS.json",
        "STAGE10_STAGE06_RELOAD_REPRO_CHECK.json",
        "STAGE10_PERTURBATION_ROBUSTNESS.csv",
        "STAGE10_INDIVIDUAL_FEATURE_ZEROOUT_SENSITIVITY.csv",
    ],
    "next":"Stage 11 — Final Evaluation Protocol / statistical comparison",
}

(OUT/"STAGE10_FINAL_MANIFEST.json").write_text(
    json.dumps(final_manifest,indent=2)
)

print("="*110)
print("QML-SLEEPNET STAGE 10 MODEL ROBUSTNESS — COMPLETE")
print("="*110)
print("Training performed: NO")
print("Model changed: NO")
print("Headline metric retuned: NO")
print("Final frozen Task-A SHA:",EXPECTED_FINAL_SHA)
print("NEXT: Stage 11 — Final Evaluation Protocol / statistical comparison")
print("Evidence:",OUT)


QML-SLEEPNET STAGE 10 MODEL ROBUSTNESS — COMPLETE
Training performed: NO
Model changed: NO
Headline metric retuned: NO
Final frozen Task-A SHA: f121a79191be00a28f33e06e7dec20cc689268b10a988c52e21284c90d1e2eef
NEXT: Stage 11 — Final Evaluation Protocol / statistical comparison
Evidence: /content/drive/MyDrive/QML_SleepNet/outputs/GUIDE_EXACT_METRICMAX/STAGE10_MODEL_ROBUSTNESS_FINAL_v1


## Acceptance rule

Stage 10 is an **evaluation stage**, not a promotion gate.

Even if a robustness condition performs badly, **do not return to model development**. Report the degradation honestly as a limitation.

Send back:

1. the executed notebook, or
2. at minimum the final `STAGE10_PERTURBATION_ROBUSTNESS.csv` output and the final completion block.

Then proceed directly to:

**Stage 11 — Final Evaluation Protocol / statistical comparison.**